In [3]:
import json
import google.generativeai as genai
import time

GEMINI_API_KEY = "*****"

genai.configure(api_key=GEMINI_API_KEY)

model_name = "gemini-2.5-flash"
model = genai.GenerativeModel(model_name)

def generate_soap_note(transcript: str, max_retries=4):
    prompt = f"""You are a physician. Summarize this conversation into a concise SOAP note.

Conversation:
{transcript}

Rules:
- Only use stated facts or minimal inference.
- Use "Not documented" when missing.
- Keep fields 1 short sentence max.
- Output ONLY the JSON below — no text before/after.

{{
  "Subjective": {{
    "Chief_Complaint": "",
    "History_of_Present_Illness": ""
  }},
  "Objective": {{
    "Physical_Exam": "",
    "Observations": ""
  }},
  "Assessment": {{
    "Diagnosis": "",
    "Severity": ""
  }},
  "Plan": {{
    "Treatment": "",
    "Follow-Up": ""
  }}
}}"""

    raw_text = None

    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json",
                    temperature=0.1,
                    max_output_tokens=1500
                )
            )

            raw_text = response.text.strip()

            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

            start = raw_text.find('{')
            if start == -1:
                raise ValueError("No JSON start found")

            brace_count = 0
            end = -1
            for i in range(start, len(raw_text)):
                if raw_text[i] == '{':
                    brace_count += 1
                elif raw_text[i] == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        end = i + 1
                        break

            if end == -1:
                raise ValueError("No complete JSON found")

            json_str = raw_text[start:end]
            soap_note = json.loads(json_str)

            if "Subjective" not in soap_note or "Assessment" not in soap_note:
                raise ValueError("Missing required sections")

            return soap_note

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            if attempt < max_retries - 1:
                print("Retrying in 6 seconds...")
                time.sleep(6)
            else:
                print("All retries failed.")
                if raw_text:
                    print("Last raw response (cleaned):")
                    print(raw_text[:1200])
                return None

transcript = """> **Physician:** *Good morning, Ms. Jones. How are you feeling today?*
>
>
> **Patient:** *Good morning, doctor. I’m doing better, but I still have some discomfort now and then.*
>
> **Physician:** *I understand you were in a car accident last September. Can you walk me through what happened?*
>
> **Patient:** *Yes, it was on September 1st, around 12:30 in the afternoon. I was driving from Cheadle Hulme to Manchester when I had to stop in traffic. Out of nowhere, another car hit me from behind, which pushed my car into the one in front.*
>
> **Physician:** *That sounds like a strong impact. Were you wearing your seatbelt?*
>
> **Patient:** *Yes, I always do.*
>
> **Physician:** *What did you feel immediately after the accident?*
>
> **Patient:** *At first, I was just shocked. But then I realized I had hit my head on the steering wheel, and I could feel pain in my neck and back almost right away.*
>
> **Physician:** *Did you seek medical attention at that time?*
>
> **Patient:** *Yes, I went to Moss Bank Accident and Emergency. They checked me over and said it was a whiplash injury, but they didn’t do any X-rays. They just gave me some advice and sent me home.*
>
> **Physician:** *How did things progress after that?*
>
> **Patient:** *The first four weeks were rough. My neck and back pain were really bad—I had trouble sleeping and had to take painkillers regularly. It started improving after that, but I had to go through ten sessions of physiotherapy to help with the stiffness and discomfort.*
>
> **Physician:** *That makes sense. Are you still experiencing pain now?*
>
> **Patient:** *It’s not constant, but I do get occasional backaches. It’s nothing like before, though.*
>
> **Physician:** *That’s good to hear. Have you noticed any other effects, like anxiety while driving or difficulty concentrating?*
>
> **Patient:** *No, nothing like that. I don’t feel nervous driving, and I haven’t had any emotional issues from the accident.*
>
> **Physician:** *And how has this impacted your daily life? Work, hobbies, anything like that?*
>
> **Patient:** *I had to take a week off work, but after that, I was back to my usual routine. It hasn’t really stopped me from doing anything.*
>
> **Physician:** *That’s encouraging. Let’s go ahead and do a physical examination to check your mobility and any lingering pain.*
>
> [**Physical Examination Conducted**]
>
> **Physician:** *Everything looks good. Your neck and back have a full range of movement, and there’s no tenderness or signs of lasting damage. Your muscles and spine seem to be in good condition.*
>
> **Patient:** *That’s a relief!*
>
> **Physician:** *Yes, your recovery so far has been quite positive. Given your progress, I’d expect you to make a full recovery within six months of the accident. There are no signs of long-term damage or degeneration.*
>
> **Patient:** *That’s great to hear. So, I don’t need to worry about this affecting me in the future?*
>
> **Physician:** *That’s right. I don’t foresee any long-term impact on your work or daily life. If anything changes or you experience worsening symptoms, you can always come back for a follow-up. But at this point, you’re on track for a full recovery.*
>
> **Patient:** *Thank you, doctor. I appreciate it.*
>
> **Physician:** *You’re very welcome, Ms. Jones. Take care, and don’t hesitate to reach out if you need anything.*
>"""

soap_result = generate_soap_note(transcript)

if soap_result:
    print("Generated SOAP Note:")
    print(json.dumps(soap_result, indent=2))
else:
    print("Failed after all retries.")

Generated SOAP Note:
{
  "Subjective": {
    "Chief_Complaint": "Patient reports occasional backaches and some discomfort.",
    "History_of_Present_Illness": "Patient was involved in a rear-end car accident on September 1st, experiencing immediate neck and back pain, diagnosed with whiplash at A&E, followed by severe pain for 4 weeks, 10 physiotherapy sessions, and now reports only occasional backaches with no emotional or functional impact."
  },
  "Objective": {
    "Physical_Exam": "Physical examination revealed full range of movement in neck and back, no tenderness, and no signs of lasting damage to muscles or spine.",
    "Observations": "Not documented."
  },
  "Assessment": {
    "Diagnosis": "Whiplash injury.",
    "Severity": "Initial severe pain, now mild and occasional, with a positive recovery trajectory."
  },
  "Plan": {
    "Treatment": "No further active treatment prescribed at this time, as patient has completed physiotherapy and is recovering well.",
    "Follow-Up":